Este é o código feito na aula pelo apresentador. 

o código abaixo serve para instalar o pacote groq.

In [1]:
%pip install groq

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


importa a biblioteca os, que serve para interagir com o sistema operacional. a função environ armazena a API KEY gerada no groq, para que o objeto do tipo Groq acesse depois.

In [ ]:
import os
os.environ['GROQ_API_KEY'] = "api key"

importa o Groq para acessar os serviços necessários para construir o agente, usando a API KEY salva no environ anteriormente. cria um chat.completions que serve para requisitar uma resposta do llama, onde o conteúdo está no "content" de messages. salva a resposta e printa a primeira.

In [3]:
import os

from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

Fast language models are revolutionizing the field of natural language processing (NLP) and have numerous benefits. Here are some key reasons why they're important:

1. **Improved User Experience**: Fast language models enable applications to respond quickly to user queries, providing a seamless and interactive experience. This is particularly crucial in real-time applications, such as chatbots, voice assistants, and language translation software.
2. **Increased Productivity**: By processing and generating text rapidly, fast language models can automate tasks like content generation, sentiment analysis, and text summarization, freeing up human resources for more complex and creative tasks.
3. **Enhanced Decision-Making**: Fast language models can analyze vast amounts of text data quickly, providing insights and patterns that can inform business decisions, medical diagnoses, and other critical applications.
4. **Real-Time Language Translation**: Fast language models enable real-time lan

onde a classe do agente é criada. o método __init__ é o construtor, onde é configurada a criação (new implícito) do objeto e define quais parâmetros são necessários, além de regras adicionais para a criação, com o if self.system is note None. o método __call__ é o padrão, ou seja, é executado sempre que uma instância da classe for chamada. serve para adicionar as mensagens na lista de "messages" do agente, considerando as condições definidas. o método execute serve para solicitar um "chat.completions", sendo essa a resposta de fato do llama, assim como feito inicialmente.

In [ ]:
class Agent:
    def __init__(self, client, system):
        self.client = client
        self.system = system
        self.messages = []
        if self.system is not None: 
            self.messages.append({"role": "system", "content": self.system})
        
    def __call__(self, message=""):
        if message:
            self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result
    
    def execute(self):
        completion = client.chat.completions.create(
            messages=self.messages,
            model="llama-3.3-70b-versatile",
        )
        return completion.choices[0].message.content

a seguir, o system_prompt, que será a mensagem padrão dada ao agente, para que ele leia e entenda o que e como precisa trabalhar. é no system_prompt que fala para o llama agir de fato como um agente react, ditando as regras de funcionamento.

In [5]:
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_planet_mass:
e.g. get_planet_mass: Earth
returns weight of the planet in kg

Example session:

Question: What is the mass of Earth times 2?
Thought: I need to find the mass of Earth
Action: get_planet_mass: Earth
PAUSE

You will be called again with this:

Observation: 5.972e24

Thought: I need to multiply this by 2
Action: calculate: 5.972e24 * 2
PAUSE

You will be called again with this:

Observation: 1,1944×10e25

If you have the answer, output it as the Answer.

Answer: The mass of Earth times 2 is 1,1944×10e25.

Now it's your turn:
""".strip()

neste espaço são definidas as ferramentas que o agente deve usar, sendo a "calculate" para calcular operações aritméticas como um seria executado de fato em um código python, retornando o valor final. a ferramenta "get_planet_mass" serve para retornar a massa do planeta requisitado.

In [6]:
#tools
def calculate(operation):
    return eval(operation)

def get_planet_mass(planet) -> float:
    match planet.lower():
        case "earth":
            return 5.972e24
        case "jupiter":
            return 1.898e27
        case "mars":
            return 6.39e23
        case "mercury":
            return 3.285e23
        case "neptune":
            return 1.024e26
        case "saturn":  
            return 5.683e26
        case "uranus":
            return 8.681e25
        case "venus":
            return 4.867e24
        case _:
            raise 0.0

instanciação da classe "Agent", criando um agente chamado neil_tyson.

In [7]:
neil_tyson = Agent(client, system_prompt)

após a inicialização do agente, dado o system_prompt com a forma como ele deve trabalhar, já podemos usá-lo. chamando o neil, chamada o método __call__ que possui o completions, e o resultado da requisição (result) é printado.

In [8]:
result = neil_tyson("What is the mass of the Earth times 5?")
print(result)

Thought: To find the mass of the Earth times 5, I first need to determine the mass of the Earth. 

Action: get_planet_mass: Earth
PAUSE


verifica quais são as mensagens existentes na lista de mensagens do neil.

In [9]:
neil_tyson.messages

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE\n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this:\n\nObservation: 1,1944×10e25\n\nIf you have the a

In [10]:
result = neil_tyson()
print(result)

uma observação serve para informar o valor retornado pela ferramenta usada.

In [11]:
observation = get_planet_mass("Earth")
print(observation)

5.972e+24


configura a observation para ser de fato enviada ao agente, de acordo com o padrão definido no system_prompt.

In [12]:
next_prompt = f"Observation: {observation}"
result = neil_tyson(next_prompt)
print(result)

Thought: Now that I have the mass of the Earth, I need to multiply this value by 5 to find the result.

Action: calculate: 5.972e+24 * 5
PAUSE


verifica novamente se o resultado da requisição passada foi adicionada a lista de mensagens do neil.

In [13]:
neil_tyson.messages

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE\n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this:\n\nObservation: 1,1944×10e25\n\nIf you have the a

In [14]:
result = neil_tyson()
print(result)

simula novamente o uso das ferramentas, calculando a massa da terra multiplicado por 5.

In [15]:
observation = calculate("3.285e23 * 5")
print(observation)

1.6425e+24


configura novamente a observação no padrão definido no system_prompt.

In [16]:
next_prompt = f"Observation: {observation}"
result = neil_tyson(next_prompt)
print(result)

Thought: The calculation result is the mass of the Earth times 5, so I have my answer.

Answer: The mass of the Earth times 5 is 2.986e+25


verifica novamente as mensagens adicionadas na lista de mensagens do neil.

In [17]:
neil_tyson.messages

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE\n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this:\n\nObservation: 1,1944×10e25\n\nIf you have the a

apenas um teste para verificar/demonstrar que a formatação do findall estava como esperada.

In [18]:
import re
result = """
Action: calculate: 5.972e24 * 5
PAUSE
"""
action = re.findall(r"Action: ([a-z]+): (.+)", result, re.IGNORECASE)

printa o resultado do teste anterior.

In [19]:
action

[('calculate', '5.972e24 * 5')]

esta é a parte final da construção do agente, onde configura todas as requisições (observations, uso de ferramentas) que fizemos de forma manual até agora, para que o agente funcione de forma automática, sem que precisemos interferir. a função agent_loop é a que será usada para executar a automação. como parâmetro, é dada a quantidade máxima de iterações, que limita o loop, para não ficar infinito, dando também o system (a mensagem padrão, system_prompt) e a query, que é a pergunta que o agente deve responder. no início, invoca o agente para saber a resposta para o "next_prompt". se existir as palavras PAUSE e Action na resposta, então é filtrado a ação e o parâmetro referente a ela. o if chosen_tool in tools serve para confirmar se a ferramenta solicitada existe, e caso exista, recupera o valor resultante da ação, fornecendo como uma observation, assim como estávamos fazendo manualmente, atualizando também o next prompt para manter a automação. por fim, verifica se existe a palavra "Answer" no último resultado, se existir, isso significa que o agente já chegou na resposta e deve terminar a sua atuação 

In [22]:
import re

def agent_loop(max_iterations, system, query):
    agent = Agent(client, system)
    tools = ['calculate', 'get_planet_mass']
    next_prompt = query
    i = 0
    while i < max_iterations:
        i += 1
        result = agent(next_prompt)
        print(result)

        if "PAUSE" in result and "Action" in result:
            action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
            chosen_tool = action[0][0]
            arg = action[0][1]

            if chosen_tool in tools:
                result_tool = eval(f"{chosen_tool}('{arg}')")
                next_prompt = f"Observation: {result_tool}"

            else:
                next_prompt = "Observation: Tool not found"

            print(next_prompt)
            continue

        if "Answer" in result:
            break


a execução agent_loop, onde acontecem todas as regras definidas anteriormente.

In [23]:
agent_loop(max_iterations=10, system=system_prompt, query="What is the mass of the Earth plus the mass of Mercury, and all of it times 5?")

Thought: To calculate the mass of the Earth plus the mass of Mercury, and all of it times 5, I first need to find the mass of Earth and the mass of Mercury. Then I can add them together and multiply by 5. I'll start by finding the mass of Earth.

Action: get_planet_mass: Earth
PAUSE
Observation: 5.972e+24
Thought: Now that I have the mass of Earth, I need to find the mass of Mercury. Once I have both, I can add them together and then multiply by 5.

Action: get_planet_mass: Mercury
PAUSE
Observation: 3.285e+23
Thought: I now have the masses of both Earth and Mercury. I can add these two values together to get the combined mass, and then multiply the result by 5 to get the final answer.

Action: calculate: (5.972e+24 + 3.285e+23) * 5
PAUSE
Observation: 3.1502500000000004e+25
Thought: I have now calculated the mass of the Earth plus the mass of Mercury, and all of it times 5. The result is the observation from the previous step.

Answer: The mass of the Earth plus the mass of Mercury, an